# Groundhog — a quantitative teardown 🔬
### The same-month vs other-month control · the Lo t-stat · no decay · the cost sweep

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Genuinely same-month seasonal?: Confirmed](https://img.shields.io/badge/Genuinely_same--month_seasonal%3F-Confirmed-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We confirm return seasonality is real and same-month-specific, then ask what survives turnover.

> ⚠️ **Not investment advice.** 398 S&P 500 names with ≥20y history (Yahoo), 2000–2026; survivorship-biased, large-cap. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (groundhog/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from groundhog import data, strategy as st
ret = data.fetch_panel()                       # cache-first; built by examples/verify.py --fetch
same = st.seasonal_hedge(ret, same_month=True)
ctrl = st.seasonal_hedge(ret, same_month=False)


C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\48-groundhog\groundhog\strategy.py:43: RuntimeWarning: Mean of empty slice
  pred = np.nanmean(R[rows, :], axis=0)


C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\48-groundhog\groundhog\strategy.py:43: RuntimeWarning: Mean of empty slice
  pred = np.nanmean(R[rows, :], axis=0)


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **Real** | +7.3%/yr, Sharpe 0.81, Lo t 4.1, no decay |
| Tradability | **Fragile** | break-even 38 bp but monthly full turnover + short book |
| Same-month seasonal? | **Confirmed** | other-month control earns −3.3%/yr |

> 💡 *In plain words:* a genuinely strange effect that passes every guard.

## 1 · The claim, steelmanned

- **H₁:** the same-month long-short earns a significant positive return.
- **H₂:** it's specific to the same month (the other-month control fails).
- **H₃:** it survives realistic costs.

## 2 · So what? — what rides on each

If H₁/H₂ hold, calendar seasonality is a real, decorrelated alpha and a puzzle for efficient markets. H₃ decides whether it's investable or merely true.

## 3 · How we'd know — the protocol

Score by same-month history → long-short → Lo t-stat → the other-month control → decade split → cost sweep and break-even.

## 4 · The teardown

### 4.1 The effect and the control

In [2]:
import pandas as pd
display(pd.DataFrame({'same-month':st.stats(same),'control (other months)':st.stats(ctrl)}).T[['mean_ann','sharpe','tstat','hit_rate','n']].round(3))

,mean_ann,sharpe,tstat,hit_rate,n
same-month,0.073,0.805,4.090,0.638,318.0
control (other months),-0.033,-0.212,-1.088,0.509,318.0


> 💡 *In plain words:* +7.3%/yr at t 4.1 for the same month; the control is negative. **H₁ and H₂ hold** — real, and specifically seasonal.

### 4.2 No decay

In [3]:
for lab,sl in [('1999-2012',same.loc[:'2012']),('2013-on',same.loc['2013':])]:
    print(f'{lab}: Sharpe {st.stats(sl)["sharpe"]:+.2f}, mean {st.stats(sl)["mean_ann"]:+.2%}/yr')

1999-2012: Sharpe +0.81, mean +8.32%/yr
2013-on: Sharpe +0.81, mean +6.37%/yr


> 💡 *In plain words:* identical across halves — the hallmark of a real effect, not a mined one.

### 4.3 The cost sweep — what survives turnover

In [4]:
rows={'gross':st.stats(same)['sharpe']}
for c in (5,10,20,30): rows[f'{c}bp']=st.stats(st.net_of_cost(same,c))['sharpe']
import pandas as pd; display(pd.Series(rows, name='net Sharpe').round(3))
print(f'break-even ≈ {st.breakeven_cost_bps(same):.0f} bp/trade')

gross    0.805
5bp      0.700
10bp     0.594
20bp     0.383
30bp     0.172
Name: net Sharpe, dtype: float64

break-even ≈ 38 bp/trade


> 💡 *In plain words:* clears realistic large-cap costs (net 0.59 at 10 bp, break-even 38 bp) — but the sweep ignores short-borrow, full monthly turnover and the survivorship flatter. **H₃ partially holds → Fragile.**

## 5 · The verdict

H₁/H₂ hold, H₃ partial → Signal `REAL`, seasonality `CONFIRMED`, Tradability `FRAGILE`.

## 6 · Could you trade it?

Plausibly, as a market-neutral overlay — but the practical drag (short borrow, monthly turnover, capacity) and the survivorship flatter mean the realised net is uncertain. It's the rare *real* one here; the honest caveat is the implementation, not the signal.

## 7 · Going further

Forks: (a) a point-in-time / small-cap universe (the effect is stronger but less tradable); (b) the seasonal *factor* (Keloharju-Linnainmaa-Nyberg 2016) vs stock-level; (c) a long-only tilt to drop the short book. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).